In [ ]:
# Lab type: prompt
# Course: EDA — Exploratory Data Analysis
# Lesson: Communicating EDA Findings
# Task: An EDA has already been run on the orders dataset and printed below.
#       Write a prompt asking an AI to draft a findings brief from this summary,
#       then audit the brief against the six-item checklist in Phase 3.

## Setup

Run this cell. It generates the orders dataset, runs the EDA, and prints a
structured summary. Copy the full output (starting from the first `=` separator
line) into your AI tool in Phase 1.

In [ ]:
!pip install pandas matplotlib seaborn numpy scipy --quiet

import pandas as pd
import numpy as np
from scipy import stats

# ── Dataset generation (same seed and structure as Lesson 07) ─────────────
rng = np.random.default_rng(42)
n = 50_000

channels = rng.choice(
    ["online", "retail", "wholesale", "enterprise"],
    size=n, p=[0.48, 0.33, 0.14, 0.05]
)
regions_8 = ["North", "South", "East", "West",
             "Northeast", "Northwest", "Southeast", "Southwest"]
region_raw = rng.choice(regions_8 + [None], size=n,
                        p=[0.14, 0.13, 0.14, 0.12, 0.12, 0.11, 0.11, 0.11, 0.02])

status_probs = {
    "online":     [0.68, 0.12, 0.08, 0.07, 0.03, 0.02],
    "retail":     [0.72, 0.11, 0.07, 0.06, 0.03, 0.01],
    "wholesale":  [0.74, 0.10, 0.07, 0.05, 0.03, 0.01],
    "enterprise": [0.73, 0.11, 0.08, 0.04, 0.03, 0.01],
}
statuses = ["delivered", "shipped", "pending", "cancelled", "returned", "processing"]
order_status = np.empty(n, dtype=object)
for ch, probs in status_probs.items():
    mask = channels == ch
    order_status[mask] = rng.choice(statuses, size=mask.sum(), p=probs)

base_date = pd.Timestamp("2022-01-01")
order_days = rng.integers(0, 730, size=n)
order_dates = [str((base_date + pd.Timedelta(days=int(d))).date()) for d in order_days]
delivery_lag = rng.integers(2, 14, size=n)
delivery_dates = []
for i, (od, st) in enumerate(zip(order_dates, order_status)):
    if st in ("pending", "processing"):
        delivery_dates.append(None)
    else:
        d = pd.Timestamp(od) + pd.Timedelta(days=int(delivery_lag[i]))
        delivery_dates.append(str(d.date()))

n_products = 500
product_weights = np.zeros(n_products)
product_weights[:50]  = 0.60 / 50
product_weights[50:]  = 0.40 / 450
product_weights = product_weights / product_weights.sum()
product_ids_pool = [f"PROD-{i:04d}" for i in range(1, n_products + 1)]
product_ids = rng.choice(product_ids_pool, size=n, p=product_weights)

quantity = rng.integers(1, 8, size=n).astype(float)
quantity[channels == "wholesale"]  = rng.integers(10, 150, size=(channels == "wholesale").sum())
quantity[channels == "enterprise"] = rng.integers(25, 500, size=(channels == "enterprise").sum())
quantity = quantity.astype(int)

channel_price_range = {
    "online": (12, 85), "retail": (15, 120),
    "wholesale": (8, 45), "enterprise": (50, 300),
}
unit_price = np.empty(n)
for ch, (lo, hi) in channel_price_range.items():
    mask = channels == ch
    unit_price[mask] = rng.uniform(lo, hi, size=mask.sum())
unit_price = unit_price.round(2)

discount = rng.beta(1.5, 8, size=n).round(3)
discount[channels == "enterprise"] = rng.beta(3, 5, size=(channels == "enterprise").sum()).round(3)

shipping_cost = (rng.uniform(3, 12, size=n) + quantity * rng.uniform(0.05, 0.35, size=n)).round(2)
shipping_cost = np.clip(shipping_cost, 2.0, 250.0)

return_raw = []
for st in order_status:
    if st == "returned":
        return_raw.append(rng.choice(["True", "Yes"]))
    else:
        return_raw.append(rng.choice(["False", "No"]))
return_flag = np.array(return_raw)

revenue = (unit_price * quantity * (1 - discount) - shipping_cost).round(2)
revenue = np.maximum(revenue, 0.5)

order_ids    = [f"ORD-{i:06d}" for i in range(1, n + 1)]
customer_ids = [f"CUST-{rng.integers(1, 10000):05d}" for _ in range(n)]

df = pd.DataFrame({
    "order_id":      order_ids,
    "customer_id":   customer_ids,
    "order_date":    order_dates,
    "product_id":    product_ids,
    "quantity":      quantity,
    "unit_price":    unit_price,
    "discount":      discount,
    "shipping_cost": shipping_cost,
    "region":        region_raw,
    "channel":       channels,
    "status":        order_status,
    "delivery_date": delivery_dates,
    "return_flag":   return_flag,
    "revenue":       revenue,
})

# ── EDA computations ──────────────────────────────────────────────────────

CH_ORDER = ["online", "retail", "wholesale", "enterprise"]

# Section 1: Data quality
missing_counts = df.isnull().sum()
missing_pcts   = (missing_counts / len(df) * 100).round(1)
dup_count      = df["order_id"].duplicated().sum()

ch_props  = pd.Series({c: (channels == c).mean() for c in CH_ORDER})
miss_ch   = (
    df[df["region"].isnull()]["channel"]
    .value_counts()
    .reindex(ch_props.index, fill_value=0)
)
_, chi2_p = stats.chisquare(
    miss_ch.values,
    f_exp=(ch_props * missing_counts["region"]).values,
)

pending_mask    = df["status"].isin(["pending", "processing"])
null_in_pending = df.loc[pending_mask, "delivery_date"].isnull().mean() * 100

# Section 2: Numeric distributions
rev_med  = df["revenue"].median()
rev_mean = df["revenue"].mean()
rev_skew = df["revenue"].skew()
rev_min  = df["revenue"].min()
rev_max  = df["revenue"].max()

up_med  = df["unit_price"].median()
up_mean = df["unit_price"].mean()
up_skew = df["unit_price"].skew()

qty_low  = (df["quantity"] <= 7).mean() * 100
qty_high = (df["quantity"] >= 10).mean() * 100

# Section 3: Revenue by channel
ch_stats = (
    df.groupby("channel")["revenue"]
    .agg(median="median", mean="mean", std="std", n="count")
    .loc[CH_ORDER]
)

# Section 4: Correlations
corr   = df[["quantity", "unit_price", "discount", "shipping_cost", "revenue"]].corr()
sc_qty = corr.loc["shipping_cost", "quantity"]
up_rev = corr.loc["unit_price",    "revenue"]
di_rev = corr.loc["discount",      "revenue"]

ch_sc_qty = {
    ch: df.loc[df["channel"] == ch, ["shipping_cost", "quantity"]].corr().iloc[0, 1]
    for ch in CH_ORDER
}

# Section 5: Temporal analysis
df_t = df.copy()
df_t["order_dt"] = pd.to_datetime(df_t["order_date"])
df_t["month"]    = df_t["order_dt"].dt.month
df_t["dow"]      = df_t["order_dt"].dt.dayofweek

_, p_month = stats.f_oneway(*[df_t.loc[df_t["month"] == m, "revenue"].values for m in range(1, 13)])
_, p_dow   = stats.f_oneway(*[df_t.loc[df_t["dow"]   == d, "revenue"].values for d in range(7)])

# Section 6: Categorical patterns
top50_skus = df["product_id"].value_counts().head(50).index
top50_vol  = df["product_id"].isin(top50_skus).mean() * 100

is_ret      = df["return_flag"].isin(["True", "Yes"])
ret_overall = is_ret.mean() * 100
ret_by_ch   = (
    df.assign(ret=is_ret).groupby("channel")["ret"].mean() * 100
).loc[CH_ORDER]

status_dist = (df["status"].value_counts(normalize=True) * 100).round(1)

# ── Print formatted EDA summary ───────────────────────────────────────────
SEP = "=" * 67
print(SEP)
print(" EDA FINDINGS SUMMARY -- Orders Dataset")
print(f" {len(df):,} rows x 14 columns")
print(SEP)

print("\nSECTION 1 -- DATA QUALITY")
print("  Missing values:")
n_reg = int(missing_counts["region"])
p_reg = float(missing_pcts["region"])
n_del = int(missing_counts["delivery_date"])
p_del = float(missing_pcts["delivery_date"])
print(f"    region        : {n_reg:>5,}  ({p_reg:.1f}%) -- uniform across channels (chi-sq p={chi2_p:.2f})")
print(f"    delivery_date : {n_del:>5,}  ({p_del:.1f}%) -- {null_in_pending:.0f}% overlap with status=pending/processing")
print("    All other columns: 0 missing")
print(f"  Duplicate order_id values: {dup_count}")
print("  return_flag encoding: mixed strings ('True'/'Yes' and 'False'/'No') -- needs conversion")

print("\nSECTION 2 -- NUMERIC DISTRIBUTIONS")
print("  revenue:")
print(f"    Median  ${rev_med:>8,.2f}  |  Mean  ${rev_mean:>8,.2f}  |  Skew  {rev_skew:.2f}")
print(f"    Range:  ${rev_min:,.2f} -- ${rev_max:,.2f}")
print("  unit_price:")
print(f"    Median  ${up_med:>8,.2f}  |  Mean  ${up_mean:>8,.2f}  |  Skew  {up_skew:.2f}")
print("  quantity  (bimodal):")
print(f"    Retail/online group   (1-7 units) : {qty_low:.1f}% of rows")
print(f"    Wholesale/enterprise  (>=10 units): {qty_high:.1f}% of rows")

print("\nSECTION 3 -- REVENUE BY CHANNEL")
print(f"  {'Channel':<12} {'Median':>9} {'Mean':>9} {'Std':>9} {'n':>7}")
for ch, row in ch_stats.iterrows():
    print(f"  {ch:<12} ${row['median']:>8,.2f} ${row['mean']:>8,.2f} ${row['std']:>8,.2f} {int(row['n']):>7,}")

print("\nSECTION 4 -- CORRELATIONS (Pearson)")
print("  Dataset-level:")
print(f"    shipping_cost ~ quantity : r = {sc_qty:.2f}")
print(f"    unit_price    ~ revenue  : r = {up_rev:.2f}")
print(f"    discount      ~ revenue  : r = {di_rev:.2f}")
print("  shipping_cost ~ quantity by channel:")
for ch in CH_ORDER:
    marker = "  <-- drives the dataset-level correlation" if ch == "wholesale" else ""
    print(f"    {ch:<12} r = {ch_sc_qty[ch]:.2f}{marker}")

print("\nSECTION 5 -- TEMPORAL ANALYSIS")
print(f"  Checked: monthly revenue seasonality  -->  no pattern  (ANOVA p={p_month:.2f})")
print(f"  Checked: day-of-week revenue effects  -->  no pattern  (ANOVA p={p_dow:.2f})")

print("\nSECTION 6 -- CATEGORICAL PATTERNS")
print(f"  product_id:  500 unique SKUs; top 50 (10%) account for {top50_vol:.1f}% of volume")
print(f"  return rate: {ret_overall:.1f}% overall")
print(f"               by channel: online={ret_by_ch['online']:.1f}%  retail={ret_by_ch['retail']:.1f}%  wholesale={ret_by_ch['wholesale']:.1f}%  enterprise={ret_by_ch['enterprise']:.1f}%")
print("  order status:")
for st in ["delivered", "shipped", "pending", "cancelled", "returned", "processing"]:
    if st in status_dist.index:
        print(f"    {st:<14}: {status_dist[st]:.1f}%")

print()
print(SEP)
print(" Copy all output above (starting from the first = line) into your AI tool.")
print(SEP)

---
## The Task

You are preparing features for a **revenue prediction model**. A completed EDA
has just been printed above. Your goal:

1. Write a prompt asking an AI to draft a findings brief from that summary
2. Paste the AI output into Phase 2
3. Audit the brief against the checklist in Phase 3

A findings brief answers exactly three questions:
1. **What surprised you?**
2. **What matters for the next step** (feature engineering for the revenue model)?
3. **What remains uncertain?**

---
## Phase 1 — Write Your Prompt

Write a prompt that gives the AI the EDA summary and instructs it to draft a
findings brief. A weak prompt returns a generic statistical recap. A strong
prompt specifies what the brief must answer, how uncertainty should be expressed,
and what the downstream use case is.

Your prompt must include at minimum:
- The EDA summary (paste the printed output above)
- The modelling goal: feature engineering for a revenue prediction model
- The three-question structure the brief must follow
- An instruction to use qualifying language for distributional claims
  ("appears right-skewed" — not "is right-skewed")

In [ ]:
# PROMPT ENTRY
#
# Write your prompt in the string below, then copy it (with the EDA summary
# pasted in) into your AI tool.
#
# Tip: structure your instructions in two parts -- first paste the EDA summary,
# then on a new line write your instructions for what the brief should contain.

PROMPT = '''
[Paste the EDA summary here]

[Write your instructions here]
'''

print(PROMPT)

---
## Phase 2 — Paste the AI Output

Double-click this cell to edit it, then replace the placeholder below with the
full text of the AI-generated findings brief. Keep the horizontal rules so the
output is clearly scoped.

---

**[Paste AI-generated findings brief here — replace this line with the full output]**

---

---
## Phase 3 — Structured Audit

For each item, mark `[ ]` as `[✓]` (pass) or `[✗]` (fail) and add a one-line
note. If you do not yet have AI output in Phase 2, go back and complete that
first.

In [ ]:
# AUDIT CHECKLIST
# ─────────────────────────────────────────────────────────────────────────
#
# 1. CERTAINTY LANGUAGE
#    [ ] Every distributional claim uses qualifying language ("appears",
#        "the data suggests", "in this sample") -- no bare assertions
#        ("revenue is right-skewed", "enterprise orders are larger")
#    Note:
#
# 2. NULL FINDINGS DOCUMENTED
#    [ ] Both Section 5 results (no monthly seasonality, no day-of-week
#        effect) appear as explicit findings -- documented as "checked and
#        found nothing", not silently omitted
#    Note:
#
# 3. STRUCTURAL MISSINGNESS
#    [ ] The delivery_date missingness is described as structural (100%
#        overlap with pending/processing status) -- not stated as a bare
#        percentage ("X% missing -- consider imputation")
#    Note:
#
# 4. CORRELATION SCOPE
#    [ ] The shipping_cost/quantity correlation is attributed to the
#        wholesale channel -- it is not presented as a universal relationship
#        across all orders
#    Note:
#
# 5. ENTERPRISE SAMPLE CONFIDENCE
#    [ ] The enterprise revenue comparison is caveated by the smaller sample
#        size (approximately one-tenth the size of the online group) -- the
#        finding is not stated as definitive
#    Note:
#
# 6. THREE-QUESTION STRUCTURE
#    [ ] The brief explicitly addresses all three: (a) what was surprising,
#        (b) what matters for next steps (feature engineering), and
#        (c) what remains uncertain
#    Note:
#
# ─────────────────────────────────────────────────────────────────────────
# FINDINGS LOG
#
# For each [x] item, record:
#   Category: Overstated certainty | Fabricated significance |
#             Omitted null finding  | Missing business context
#   Issue:    what the AI got wrong or omitted
#   Fix:      write corrected text in Phase 4
#
# ITEM 1:
# ITEM 2:
# ITEM 3:
# ─────────────────────────────────────────────────────────────────────────

---
## Phase 4 — Record Corrections

For each `[✗]` audit item, write the corrected finding text below. Label each
block with the audit item number it addresses.

In [ ]:
# CORRECTED FINDINGS
# Label each block with the audit item number (e.g. '# --- Item 3 ---')

# --- Item X ---

In [ ]:
# INSTRUCTOR NOTE
# Expected failure modes from naive AI prompts for this task.
# Use this to verify the lab is surfacing the intended issues.
#
# 1. CERTAINTY LANGUAGE  (very consistent)
#    AI findings briefs almost always use bare assertions: "Revenue is strongly
#    right-skewed", "Enterprise channel generates significantly higher revenue."
#    No hedging, no "in this sample", no "appears". Students should look for
#    any distributional claim stated as settled fact rather than a finding from
#    this specific dataset and sample.
#
# 2. NULL FINDINGS OMITTED  (very common)
#    AI tools report what was found, not what was checked and not found. The
#    Section 5 results (no monthly seasonality, no day-of-week effect) are
#    almost never included in AI-generated briefs. This matters for modelling:
#    the absence of a temporal signal means month/DOW features can be
#    deprioritised. An AI that omits this leaves the modelling engineer without
#    that guidance.
#
# 3. STRUCTURAL MISSINGNESS  (common)
#    A typical AI brief says "delivery_date has X% missing values -- consider
#    imputation or dropping." It does not note that 100% of these nulls
#    correspond to pending/processing orders, making imputation wrong. Correct
#    framing: "delivery_date is null for all in-progress orders -- this is by
#    design, not a data quality issue. Do not impute."
#
# 4. CORRELATION SCOPE  (common with weaker prompts)
#    The dataset-level r=0.38 between shipping_cost and quantity is
#    wholesale-driven (r>0.65 within wholesale, r<0.20 elsewhere). AI briefs
#    typically present the dataset-level correlation as a general finding and
#    recommend including shipping_cost as a correlated feature -- without
#    noting this is a channel composition effect. Within online and retail
#    orders the relationship is weak.
#
# 5. ENTERPRISE SAMPLE SIZE  (common)
#    "Enterprise orders generate 4-5x more revenue" is stated flatly. The
#    enterprise group (n~2,500) is roughly one-tenth the size of the online
#    group (n~24,000). AI tools rarely caveat summary statistics with
#    confidence considerations based on sample size.
#
# 6. THREE-QUESTION STRUCTURE  (varies by prompt quality)
#    "What remains uncertain" is the question most often omitted. AI briefs
#    tend to present all findings as resolved. Typical omissions:
#    - Whether the quantity bimodality reflects two customer segments or
#      two data collection systems (not checked in this EDA)
#    - Whether the enterprise return rate (higher than other channels) reflects
#      real behaviour or a return_flag encoding artefact (mixed True/Yes strings)
#    - Whether the top-50 SKU concentration is stable over time or a
#      recent shift (temporal check not performed)